In [1]:
import pandas as pd

tracks = pd.read_csv(r"C:\Users\moero\Downloads\fma_metadata\tracks.csv", index_col=0, header=[0,1])
genres = pd.read_csv(r"C:\Users\moero\Downloads\fma_metadata\genres.csv", index_col=0)
features = pd.read_csv(r"C:\Users\moero\Downloads\fma_metadata\features.csv", index_col=0, header=[0,1,2])

print("tracks shape:", tracks.shape)
print("genres shape:", genres.shape)
print("features shape:", features.shape)

tracks shape: (106574, 52)
genres shape: (163, 4)
features shape: (106574, 518)


In [2]:
# 1. Get the top-level genre label for each track
# In FMA, the 'track' / 'genre_top' column holds a single main genre per track
genre_top = tracks[('track', 'genre_top')]

# 2. Keep only tracks that have a known top genre (drop rows with NaN)
genre_top = genre_top.dropna()

print("Number of tracks with a top genre:", genre_top.shape[0])
print("First 10 genre labels:", genre_top.unique()[:10])

Number of tracks with a top genre: 49598
First 10 genre labels: ['Hip-Hop' 'Pop' 'Rock' 'Experimental' 'Folk' 'Jazz' 'Electronic' 'Spoken'
 'International' 'Soul-RnB']


In [3]:
# Make sure features and labels use the same track index
# Intersect the indices that appear in both features and genre_top
common_idx = features.index.intersection(genre_top.index)

# Subset features and labels to the common tracks
X_full = features.loc[common_idx]
y_full = genre_top.loc[common_idx]

print("Common tracks:", len(common_idx))
print("X_full shape:", X_full.shape)
print("y_full shape:", y_full.shape)
print("First 10 labels in y_full:", y_full.unique()[:10])

Common tracks: 49598
X_full shape: (49598, 518)
y_full shape: (49598,)
First 10 labels in y_full: ['Hip-Hop' 'Pop' 'Rock' 'Experimental' 'Folk' 'Jazz' 'Electronic' 'Spoken'
 'International' 'Soul-RnB']


In [4]:
import numpy as np

# Choose a subset of genres to focus on
selected_genres = [
    'Hip-Hop', 'Pop', 'Rock', 'Experimental',
    'Folk', 'Jazz', 'Electronic', 'Soul-RnB'
]

# Keep only tracks whose genre is in this list
mask = y_full.isin(selected_genres)
X_sel = X_full[mask]
y_sel = y_full[mask]

print("Shape after genre filtering:", X_sel.shape)
print("Number of tracks per genre:")
print(y_sel.value_counts())

# Optionally, downsample to at most 1000 tracks per genre to keep it fast
max_per_genre = 1000
indices = []

for g in selected_genres:
    g_idx = y_sel[y_sel == g].index
    if len(g_idx) > max_per_genre:
        g_idx = np.random.choice(g_idx, max_per_genre, replace=False)
    indices.extend(g_idx)

X_small = X_sel.loc[indices]
y_small = y_sel.loc[indices]

print("Final X_small shape:", X_small.shape)
print("Final y_small distribution:")
print(y_small.value_counts())

Shape after genre filtering: (43595, 518)
Number of tracks per genre:
(track, genre_top)
Rock            14182
Experimental    10608
Electronic       9372
Hip-Hop          3552
Folk             2803
Pop              2332
Jazz              571
Soul-RnB          175
Name: count, dtype: int64
Final X_small shape: (6746, 518)
Final y_small distribution:
(track, genre_top)
Hip-Hop         1000
Pop             1000
Rock            1000
Experimental    1000
Folk            1000
Electronic      1000
Jazz             571
Soul-RnB         175
Name: count, dtype: int64


In [5]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, classification_report

# Train/test split (stratify to preserve genre proportions)
X_train, X_test, y_train, y_test = train_test_split(
    X_small, y_small, test_size=0.2, random_state=42, stratify=y_small
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (5396, 518)
Test shape: (1350, 518)


In [6]:
from sklearn.linear_model import LogisticRegression

# Logistic Regression with scaling in a pipeline
logreg_clf = make_pipeline(
    StandardScaler(with_mean=True, with_std=True),
    LogisticRegression(
        max_iter=1000,
        multi_class='multinomial',
        n_jobs=-1
    )
)

logreg_clf.fit(X_train, y_train)
y_pred_logreg = logreg_clf.predict(X_test)
logreg_acc = accuracy_score(y_test, y_pred_logreg)

print("Logistic Regression accuracy:", logreg_acc)

Logistic Regression accuracy: 0.5459259259259259


In [7]:
from sklearn.neighbors import KNeighborsClassifier

knn_results = {}

for k in [3, 5, 7, 9]:
    knn_clf = make_pipeline(
        StandardScaler(with_mean=True, with_std=True),
        KNeighborsClassifier(n_neighbors=k)
    )
    knn_clf.fit(X_train, y_train)
    y_pred_knn = knn_clf.predict(X_test)
    acc = accuracy_score(y_test, y_pred_knn)
    knn_results[k] = acc
    print(f"KNN accuracy (k={k}): {acc}")

# Pick the best k
best_k = max(knn_results, key=knn_results.get)
best_knn_acc = knn_results[best_k]

print("\nBest KNN k:", best_k)
print("Best KNN accuracy:", best_knn_acc)

KNN accuracy (k=3): 0.46074074074074073
KNN accuracy (k=5): 0.4792592592592593
KNN accuracy (k=7): 0.4948148148148148
KNN accuracy (k=9): 0.48444444444444446

Best KNN k: 7
Best KNN accuracy: 0.4948148148148148


In [8]:
import pandas as pd

results_df = pd.DataFrame({
    "Model": ["Logistic Regression", "KNN (k=7)"],
    "Test Accuracy": [logreg_acc, best_knn_acc]
})

results_df

,Model,Test Accuracy
0,Logistic Regression,0.545926
1,KNN (k=7),0.494815


In [9]:
print("Logistic Regression classification report:")
print(classification_report(y_test, y_pred_logreg))

# Refit best KNN to get its predictions
best_knn_clf = make_pipeline(
    StandardScaler(with_mean=True, with_std=True),
    KNeighborsClassifier(n_neighbors=best_k)
)
best_knn_clf.fit(X_train, y_train)
y_pred_knn_best = best_knn_clf.predict(X_test)

print("\nBest KNN (k={}) classification report:".format(best_k))
print(classification_report(y_test, y_pred_knn_best))

Logistic Regression classification report:
              precision    recall  f1-score   support

  Electronic       0.50      0.51      0.51       200
Experimental       0.50      0.54      0.52       200
        Folk       0.63      0.65      0.64       200
     Hip-Hop       0.64      0.69      0.66       200
        Jazz       0.53      0.54      0.54       115
         Pop       0.35      0.28      0.31       200
        Rock       0.64      0.62      0.63       200
    Soul-RnB       0.45      0.51      0.48        35

    accuracy                           0.55      1350
   macro avg       0.53      0.54      0.54      1350
weighted avg       0.54      0.55      0.54      1350


Best KNN (k=7) classification report:
              precision    recall  f1-score   support

  Electronic       0.52      0.33      0.40       200
Experimental       0.45      0.28      0.34       200
        Folk       0.60      0.69      0.64       200
     Hip-Hop       0.51      0.73      0.60       

In [10]:
from sklearn.decomposition import PCA

pca_knn_results = {}

for n_comp in [20, 50, 100]:
    for k in [3, 5, 7]:
        pca_knn_clf = make_pipeline(
            StandardScaler(with_mean=True, with_std=True),
            PCA(n_components=n_comp, random_state=42),
            KNeighborsClassifier(n_neighbors=k)
        )
        
        pca_knn_clf.fit(X_train, y_train)
        y_pred_pca_knn = pca_knn_clf.predict(X_test)
        acc = accuracy_score(y_test, y_pred_pca_knn)
        
        pca_knn_results[(n_comp, k)] = acc
        print(f"PCA+KNN accuracy (n_components={n_comp}, k={k}): {acc}")

best_pca_knn = max(pca_knn_results, key=pca_knn_results.get)
best_pca_knn_acc = pca_knn_results[best_pca_knn]

print("\nBest PCA+KNN settings:", best_pca_knn)
print("Best PCA+KNN accuracy:", best_pca_knn_acc)

PCA+KNN accuracy (n_components=20, k=3): 0.4177777777777778
PCA+KNN accuracy (n_components=20, k=5): 0.4414814814814815
PCA+KNN accuracy (n_components=20, k=7): 0.4488888888888889
PCA+KNN accuracy (n_components=50, k=3): 0.4711111111111111
PCA+KNN accuracy (n_components=50, k=5): 0.47555555555555556
PCA+KNN accuracy (n_components=50, k=7): 0.4925925925925926
PCA+KNN accuracy (n_components=100, k=3): 0.4696296296296296
PCA+KNN accuracy (n_components=100, k=5): 0.4874074074074074
PCA+KNN accuracy (n_components=100, k=7): 0.5007407407407407

Best PCA+KNN settings: (100, 7)
Best PCA+KNN accuracy: 0.5007407407407407


In [11]:
final_results = pd.DataFrame({
    "Method": [
        "Logistic Regression",
        "KNN (k=7)",
        "PCA + KNN (100 components, k=7)"
    ],
    "Test Accuracy": [
        logreg_acc,
        best_knn_acc,
        best_pca_knn_acc
    ]
})

final_results

,Method,Test Accuracy
0,Logistic Regression,0.545926
1,KNN (k=7),0.494815
2,"PCA + KNN (100 components, k=7)",0.500741
